# 04 — UMAP Embedding Visualization

This notebook projects the MiniLM tweet embeddings from 384 dimensions down to 2D using **UMAP**, then clusters them with **K-Means** to reveal topical structure in the tweet dataset.

**What UMAP does:** It finds a 2D layout that preserves the local neighborhood structure of the high-dimensional space. Tweets that are semantically similar end up near each other in the plot.

**Prerequisites:** Run `01_build_index.ipynb` first to generate `minilm_embeddings.npy`.

In [ ]:
# --- STEP 0: INSTALL DEPENDENCIES ---
# umap-learn: the UMAP algorithm
# scikit-learn: K-Means clustering
# matplotlib: plotting
!pip install umap-learn scikit-learn matplotlib --quiet

In [ ]:
# --- STEP 1: IMPORTS ---
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import umap
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

print("Imports ready.")

In [ ]:
# --- STEP 2: LOAD MINILM EMBEDDINGS ---

base = '/content/' if os.path.exists('/content/minilm_embeddings.npy') else ''

print("Loading MiniLM embeddings...")
embeddings = np.load(f'{base}minilm_embeddings.npy')
print(f"Embeddings shape: {embeddings.shape}  ({embeddings.shape[0]:,} tweets × {embeddings.shape[1]} dims)")

with open(f'{base}tweets.json', 'r', encoding='utf-8') as f:
    tweets = json.load(f)

In [ ]:
# --- STEP 3: SAMPLE FOR SPEED (OPTIONAL) ---
# UMAP on 110k × 384 can take 10-20 minutes on CPU.
# We sample 20k tweets — enough to reveal cluster structure clearly.
# Remove the sampling to plot everything (expect longer runtime).

SAMPLE_SIZE = 20_000

if len(embeddings) > SAMPLE_SIZE:
    # Use a fixed seed so the sample is reproducible across runs
    rng = np.random.default_rng(seed=42)
    sample_idx = rng.choice(len(embeddings), size=SAMPLE_SIZE, replace=False)
    sample_idx.sort()
    emb_sample = embeddings[sample_idx]
    tweet_sample = [tweets[i] for i in sample_idx]
    print(f"Sampled {SAMPLE_SIZE:,} of {len(embeddings):,} tweets for visualization.")
else:
    emb_sample = embeddings
    tweet_sample = tweets
    print(f"Using all {len(embeddings):,} tweets (no sampling needed).")

In [ ]:
# --- STEP 4: UMAP DIMENSIONALITY REDUCTION ---

def run_umap(embeddings, n_neighbors=15, min_dist=0.1, random_state=42):
    """
    Project high-dimensional embeddings to 2D using UMAP.

    Parameters:
        embeddings (np.ndarray): Float array of shape (n, dim).
        n_neighbors (int): Controls how much local vs. global structure UMAP preserves.
                           Lower = more local clusters; higher = broader structure.
        min_dist (float): Minimum distance between points in 2D space.
                          Lower = tighter clusters; higher = more spread out.
        random_state (int): Seed for reproducibility.

    Returns:
        np.ndarray: 2D array of shape (n, 2).
    """
    reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric='cosine',  # use cosine distance — matches how we built the index
        random_state=random_state
    )
    return reducer.fit_transform(embeddings)


print("Running UMAP (this takes a few minutes on CPU)...")
embedding_2d = run_umap(emb_sample)
print(f"UMAP complete — output shape: {embedding_2d.shape}")

In [ ]:
# --- STEP 5: K-MEANS CLUSTERING ---

def cluster_embeddings(embeddings_2d, n_clusters=12, random_state=42):
    """
    Cluster the 2D UMAP projections with K-Means.

    We run K-Means on the 2D projections (not the original high-dim embeddings)
    because the goal is to color the visualization — the 2D layout already
    encodes semantic proximity, so clustering in 2D gives meaningful groups.

    Parameters:
        embeddings_2d (np.ndarray): 2D UMAP output, shape (n, 2).
        n_clusters (int): Number of topic clusters. 10-15 works well for tweet data.
        random_state (int): Seed for reproducibility.

    Returns:
        np.ndarray: Integer cluster label for each point.
    """
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init='auto')
    labels = kmeans.fit_predict(embeddings_2d)
    return labels


N_CLUSTERS = 12
print(f"Clustering into {N_CLUSTERS} groups with K-Means...")
cluster_labels = cluster_embeddings(embedding_2d, n_clusters=N_CLUSTERS)
print(f"Clustering complete. Label distribution:")
unique, counts = np.unique(cluster_labels, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  Cluster {label}: {count:,} tweets")

In [ ]:
# --- STEP 6: PLOT AND SAVE ---

def plot_umap_clusters(embedding_2d, labels, n_clusters, output_path='umap_clusters.png'):
    """
    Create and save a scatter plot of UMAP-projected tweet embeddings, colored by cluster.

    Parameters:
        embedding_2d (np.ndarray): 2D UMAP coordinates, shape (n, 2).
        labels (np.ndarray): Cluster label for each point.
        n_clusters (int): Total number of clusters (used for colormap selection).
        output_path (str): File path to save the PNG figure.

    Returns:
        None. Saves figure to output_path and displays inline.
    """
    fig, ax = plt.subplots(figsize=(14, 10))

    # Use a colormap with enough distinct colors for all clusters
    cmap = plt.cm.get_cmap('tab20', n_clusters)

    scatter = ax.scatter(
        embedding_2d[:, 0],
        embedding_2d[:, 1],
        c=labels,
        cmap=cmap,
        s=1.5,        # small dots so 20k points don't overlap too much
        alpha=0.6     # slight transparency to show density
    )

    cbar = plt.colorbar(scatter, ax=ax, ticks=range(n_clusters))
    cbar.set_label('Cluster', fontsize=12)

    ax.set_title(
        f'UMAP Projection of {len(embedding_2d):,} Tweet Embeddings\n'
        f'(MiniLM all-MiniLM-L6-v2, {n_clusters} K-Means clusters)',
        fontsize=14
    )
    ax.set_xlabel('UMAP Dimension 1', fontsize=12)
    ax.set_ylabel('UMAP Dimension 2', fontsize=12)
    ax.set_facecolor('#f8f8f8')

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Figure saved to {output_path}")


plot_umap_clusters(embedding_2d, cluster_labels, N_CLUSTERS)

In [ ]:
# --- STEP 7: SAMPLE TWEETS PER CLUSTER ---
# Print 3 example tweets from each cluster so we can understand what each group represents.
# Useful for labeling clusters in a README or portfolio description.

SAMPLES_PER_CLUSTER = 3

print(f"Sample tweets from each cluster (to help interpret the visualization):")
print("=" * 70)

for cluster_id in range(N_CLUSTERS):
    # Get indices of all tweets in this cluster
    cluster_mask = np.where(cluster_labels == cluster_id)[0]

    # Pick a few examples without replacement
    chosen = np.random.choice(cluster_mask, size=min(SAMPLES_PER_CLUSTER, len(cluster_mask)), replace=False)

    print(f"\nCluster {cluster_id} ({len(cluster_mask):,} tweets):")
    for idx in chosen:
        print(f"  - {tweet_sample[idx][:110].replace(chr(10), ' ')}")